# Fase 1 — Almacenamiento distribuido
## Ingesta y particionado de datos

**Pipeline:** CC-News + MIND Large → Parquet particionado (simulación HDFS local)

**Fuentes:**
- CC-News: ~3.5 GB, sin etiquetas → usado en Fase 3 (LSH)
- MIND Large: ~1.5 GB comprimido, 18 categorías etiquetadas → usado en Fase 4 (MLP)

## Configuración de rutas


In [1]:
import os

# El notebook vive en notebooks/ — subimos un nivel para llegar a la raíz del proyecto
PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATOS_PATH   = os.path.join(PROJECT_PATH, "datos")

# Datos crudos — MIND ya existe, cc_news se crea al descargar
MIND_TRAIN_TSV = os.path.join(DATOS_PATH, "MINDlarge_train", "news.tsv")
MIND_DEV_TSV = os.path.join(DATOS_PATH, "MINDlarge_dev",   "news.tsv")
RAW_CC = os.path.join(DATOS_PATH, "cc_news")

# Datos procesados (Parquet) — se crean al correr el notebook
PROC_CC = os.path.join(DATOS_PATH, "processed", "cc_news")
PROC_MIND = os.path.join(DATOS_PATH, "processed", "mind_large")

for path in [RAW_CC, PROC_CC, PROC_MIND]:
    os.makedirs(path, exist_ok=True)

print(f"PROJECT_PATH -> {PROJECT_PATH}")
print(f"  MIND train -> {MIND_TRAIN_TSV}  ({'OK' if os.path.exists(MIND_TRAIN_TSV) else 'FALTA'})")
print(f"  MIND dev   -> {MIND_DEV_TSV}  ({'OK' if os.path.exists(MIND_DEV_TSV) else 'FALTA'})")
print(f"  CC-News    -> {RAW_CC}  (se llena al descargar)")
print(f"  Procesados -> {os.path.join(DATOS_PATH, 'processed')}")

PROJECT_PATH -> /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos
  MIND train -> /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/MINDlarge_train/news.tsv  (OK)
  MIND dev   -> /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/MINDlarge_dev/news.tsv  (OK)
  CC-News    -> /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/cc_news  (se llena al descargar)
  Procesados -> /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed


## 1. Instalación de dependencias

In [2]:
# Ejecuta esta celda solo si marca que no la encuentra
# !pip install pyspark datasets

## 2. Inicialización de PySpark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

spark = (SparkSession.builder
        .appName("Ingesta-Noticias")
        .master("local[*]")
        .config("spark.driver.memory", "8g")
        .config("spark.driver.maxResultSize", "4g")
        .config("spark.sql.parquet.compression.codec", "snappy")
        .config("spark.sql.shuffle.partitions", "8")
        .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} listo.")

26/06/06 15:59:26 WARN Utils: Your hostname, MacBook-Pro-de-Milena.local resolves to a loopback address: 127.0.0.1; using 192.168.1.250 instead (on interface en0)
26/06/06 15:59:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 15:59:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark 3.5.1 listo.


## 3. CC-News — Descarga y guardado en Parquet

La descarga tarda ~15-30 min dependiendo de tu conexión. Se guarda directo en disco.

In [4]:
from datasets import load_dataset

print("Descargando CC-News desde HuggingFace...")
cc_hf = load_dataset(
    "vblagoje/cc_news",
    cache_dir=RAW_CC,
    split="train")
print(f"CC-News descargado: {len(cc_hf):,} artículos")
print("Columnas:", cc_hf.column_names)

Descargando CC-News desde HuggingFace...


26/06/06 15:59:40 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


CC-News descargado: 708,241 artículos
Columnas: ['title', 'text', 'domain', 'date', 'description', 'url', 'image_url']


In [5]:
# Guardamos CC-News a Parquet directamente con HuggingFace (sin pasar por pandas)
# Esto es eficiente en memoria porque procesa por lotes internamente
cc_parquet_raw = os.path.join(RAW_CC, "cc_news_raw.parquet")

print("Guardando CC-News a Parquet en disco...")
cc_hf.select_columns(["title", "text", "description", "url", "date", "domain"]) \
     .to_parquet(cc_parquet_raw)
print(f"Guardado en: {cc_parquet_raw}")

# Ahora Spark lee desde disco — sin cargar todo en memoria del driver
cc_df = spark.read.parquet(cc_parquet_raw)
print(f"Spark cargó {cc_df.count():,} filas desde Parquet")

Guardando CC-News a Parquet en disco...


Creating parquet from Arrow format:   0%|          | 0/709 [00:00<?, ?ba/s]

Guardado en: /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/cc_news/cc_news_raw.parquet


Spark cargó 708,241 filas desde Parquet


## Limpieza

In [6]:
# Limpieza y guardado en Parquet procesado
cc_df = (cc_df
        .filter(F.col("text").isNotNull() & (F.length(F.col("text")) > 100))
        .withColumn("source", F.lit("cc_news"))
        .withColumn("doc_id", F.monotonically_increasing_id()))

(cc_df.repartition(8)
      .write
      .mode("overwrite")
      .parquet(PROC_CC))

cc_verificado = spark.read.parquet(PROC_CC)
print(f"CC-News procesado guardado: {cc_verificado.count():,} filas → {PROC_CC}")
cc_verificado.printSchema()

CC-News procesado guardado: 703,488 filas → /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed/cc_news
root
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- description: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- source: string (nullable = true)
 |-- doc_id: long (nullable = true)



In [7]:
# Guardamos en Parquet — particionado por dominio para facilitar consultas
(cc_df.repartition(8)  # 8 particiones para simular distribucion en nodos
      .write
      .mode("overwrite")
      .parquet(PROC_CC))

print(f"CC-News guardado en Parquet: {PROC_CC}")

# Verificamos
cc_verificado = spark.read.parquet(PROC_CC)
print(f"Verificación — filas leídas: {cc_verificado.count():,}")

CC-News guardado en Parquet: /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed/cc_news
Verificación — filas leídas: 703,488


## 4. MIND Large — Carga y guardado en Parquet

MIND usa formato TSV. El archivo principal es `news.tsv` (artículos con etiquetas).

**Estructura en el repositorio:**
```
datos/
├── MINDlarge_train/
│   ├── news.tsv          ← 101,527 artículos
│   └── behaviors.tsv
└── MINDlarge_dev/
    ├── news.tsv          ← 72,023 artículos
    └── behaviors.tsv
```

In [8]:
import os

# Verificamos que los archivos existen
for path in [MIND_TRAIN_TSV, MIND_DEV_TSV]:
    existe = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if existe else 0
    print(f"{'OK' if existe else 'FALTA':5s}  {path}  ({size_mb:.1f} MB)")

OK     /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/MINDlarge_train/news.tsv  (84.9 MB)
OK     /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/MINDlarge_dev/news.tsv  (59.1 MB)


In [9]:
MIND_SCHEMA = StructType([
    StructField("news_id",           StringType(), True),
    StructField("category",          StringType(), True),
    StructField("subcategory",       StringType(), True),
    StructField("title",             StringType(), True),
    StructField("abstract",          StringType(), True),
    StructField("url",               StringType(), True),
    StructField("title_entities",    StringType(), True),
    StructField("abstract_entities", StringType(), True)])

def load_mind_news(tsv_path, split_name):
    return (
        spark.read
        .option("sep", "\t")
        .option("header", "false")
        .schema(MIND_SCHEMA)
        .csv(tsv_path)
        .withColumn("split",  F.lit(split_name))
        .withColumn("source", F.lit("mind_large"))
        .withColumn("text",   F.concat_ws(" ", F.col("title"), F.col("abstract")))
        .drop("title_entities", "abstract_entities")
        .filter(F.col("title").isNotNull() & F.col("category").isNotNull()))

mind_train = load_mind_news(MIND_TRAIN_TSV, "train")
mind_valid = load_mind_news(MIND_DEV_TSV,   "dev")

mind_df = mind_train.unionByName(mind_valid)

print(f"MIND train: {mind_train.count():,} artículos")
print(f"MIND dev:   {mind_valid.count():,} artículos")
print(f"Total MIND: {mind_df.count():,} artículos")
mind_df.printSchema()

MIND train: 101,527 artículos


MIND dev:   72,023 artículos


Total MIND: 173,550 artículos
root
 |-- news_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- title: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- url: string (nullable = true)
 |-- split: string (nullable = false)
 |-- source: string (nullable = false)
 |-- text: string (nullable = false)



In [10]:
# Distribucion de categorias
print("Distribución de categorías en MIND:")
mind_df.groupBy("category").count().orderBy(F.desc("count")).show(20, truncate=False)

Distribución de categorías en MIND:


+-------------+-----+
|category     |count|
+-------------+-----+
|sports       |53599|
|news         |52304|
|finance      |10247|
|travel       |8336 |
|lifestyle    |8015 |
|foodanddrink |7876 |
|video        |7536 |
|weather      |7046 |
|autos        |5519 |
|health       |5306 |
|tv           |2402 |
|music        |2179 |
|entertainment|1561 |
|movies       |1492 |
|kids         |125  |
|middleeast   |4    |
|games        |2    |
|northamerica |1    |
+-------------+-----+



In [11]:
# Guardamos en Parquet — particionado por category para acelerar Fase 4
(mind_df
    .repartition(4)
    .write
    .mode("overwrite")
    .partitionBy("category")
    .parquet(PROC_MIND))

print(f"MIND guardado en Parquet: {PROC_MIND}")

# Verificamos
mind_verificado = spark.read.parquet(PROC_MIND)
print(f"Verificación — filas leídas: {mind_verificado.count():,}")

MIND guardado en Parquet: /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed/mind_large
Verificación — filas leídas: 173,550


## 5. Resumen del almacenamiento

Análisis de tamaño y estructura.

In [12]:
import subprocess

def get_dir_size(path):
    result = subprocess.run(["du", "-sh", path], capture_output=True, text=True)
    return result.stdout.split()[0] if result.returncode == 0 else "N/A"

print("*-" * 25)
print("RESUMEN DE ALMACENAMIENTO")
print("*-" * 25)
print(f"CC-News Parquet:  {get_dir_size(PROC_CC):>8s}  → {PROC_CC}")
print(f"MIND Parquet:     {get_dir_size(PROC_MIND):>8s}  → {PROC_MIND}")
print("*-" * 25)

print("\nEstadísticas del corpus:")
print(f"  CC-News:  {cc_verificado.count():>8,} artículos  (sin etiquetas — para LSH)")
print(f"  MIND:     {mind_verificado.count():>8,} artículos  (18 categorías — para MLP)")

print("\nParticiones Parquet (simulación de bloques HDFS):")
cc_parts  = len(os.listdir(PROC_CC))
print(f"  CC-News:  {cc_parts} archivos Parquet")
print(f"  MIND:     particionado por categoría (1 carpeta por clase)")

*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
RESUMEN DE ALMACENAMIENTO
*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
CC-News Parquet:      1.1G  → /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed/cc_news
MIND Parquet:          64M  → /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed/mind_large
*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-

Estadísticas del corpus:


Python(13301) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(13302) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  CC-News:   703,488 artículos  (sin etiquetas — para LSH)
  MIND:      173,550 artículos  (18 categorías — para MLP)

Particiones Parquet (simulación de bloques HDFS):
  CC-News:  18 archivos Parquet
  MIND:     particionado por categoría (1 carpeta por clase)


In [13]:
spark.stop()
print("SparkSession cerrada. Fase 1 completada.")
print("Siguiente paso: 02_preprocesamiento.ipynb")

SparkSession cerrada. Fase 1 completada.
Siguiente paso: 02_preprocesamiento.ipynb
